# `03_memory.ipynb`

## Key Concept
- State에 `messages` 항목에 대한 설명 -> Graph 내부에서 턴마다 필요한 메세지(AI, Human, System, Tool)를 쌓는 용도
- `InMemorySaver`로 테스트 -> 여러 턴에 대해 저장하자
- `Postgres` 에 직접 저장하는 방법

## 추후에 메모리 데이터가 너무 커지는 것을 막으려면
- TTL(Time To Live) - 데이터의 수명을 결정해서 수명이 지나면 삭제
- `checkpoints`,`checkpoints_writes` 테이블을 주기적으로 `DELETE`
- 기존 메세지 다 날아가 버리는 문제 -> 추가로 대화 내역(내가보낸 메세지 <-> AI가 내보낸 마지막 답변)은 따로 저장
- `prune`을 사용해서, 마지막 스냅샷만 살리기

In [ ]:
from dotenv import load_dotenv
load_dotenv()

In [ ]:
# 직접 만들기 (교육용)
from typing import TypedDict, Annotated
from langgraph.graph import add_messages

class MyState(TypedDict):
    # messages: list  # 그냥 리스트임 -> 교체해야하면 교체됨
    messages: Annotated[list, add_messages]  # 교체할 타이밍에, 교체하지 않고 쌓아 나가는 기능을 추가해주세요
    is_good: bool

In [ ]:
# 앞으로 모든 state는 이렇게 만든다. -> 위의 클래스와 완전히 동일한 기능 상속받은 MessagesState에 Annotated - add_messages 기능이 포함되어 있음
from langgraph.graph import MessagesState 

class MyState(MessagesState):
    # messages 기능 자동 탑재
    is_good: bool

In [ ]:
# node
from langchain.chat_models import init_chat_model

llm = init_chat_model('openai:gpt-4.1-mini')

# is_good 처리
def node_a(state: MyState):
    return {'is_good': True}  # 기존 state의 'is_good' 을 바꿔주세요


# AI 답변 생성
def node_b(state: MyState):
    messages = state['messages']
    ai_msg = llm.invoke(messages)
    return {'messages': [ai_msg]}  # 기존 state의 'messages'를 교체해 주세요

In [ ]:
from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.checkpoint.memory import InMemorySaver #테스트용 내장메모리 사용하기위함

graph = StateGraph(MyState)
graph.add_node(node_a)
graph.add_node(node_b)
graph.add_edge(START, 'node_a')
graph.add_edge('node_a', 'node_b')
graph.add_edge('node_b', END)


In [ ]:
from langchain.messages import HumanMessage
# 테스트용 메모리
workflow = graph.compile(checkpointer=InMemorySaver())
config = {'configurable': {'thread_id': '123-456'}}  # 각 세션의 고유 id로 구분

result = workflow.invoke(
    {'messages': [HumanMessage('굿굿')]},  # 1번 인자: state
    config,  # 2번 인자, 설정값
)

In [ ]:
for msg in result['messages']:
    msg.pretty_print()

## Postgres 영구저장

In [ ]:
# 영구저장 메모리
# uv add langgraph-checkpoint-postgres psycopg[binary]
import os
from dotenv import load_dotenv
from langgraph.checkpoint.postgres import PostgresSaver ## 테스트용 내장메모리 쓰다가 최종에 이거로 교체

load_dotenv(override=True)
DB_URI = os.getenv('POSTGRES_URI')

with PostgresSaver.from_conn_string(DB_URI) as checkpointer:
    # 현재 비어있는 DB에 테이블 생성 및 초기화
    checkpointer.setup()
    pg_workflow = graph.compile(checkpointer=checkpointer)

    config = {'configurable': {'thread_id': 'qwer-9876'}}
    init_state = {'messages': [
        HumanMessage('너 나랑 지금까지 무슨 이야기 했어?'),
    ]}
    result = pg_workflow.invoke(init_state, config)

In [38]:
result

{'messages': [HumanMessage(content='너 나랑 지금까지 무슨 이야기 했어?', additional_kwargs={}, response_metadata={}, id='a941a3f8-24e2-4d20-ae8f-0e86aea5e6fb'),
  AIMessage(content='안녕하세요! 지금까지 우리는 특별한 이야기를 나누지 않았어요. 도와드릴 내용이 있으면 언제든지 말씀해 주세요!', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 31, 'prompt_tokens': 18, 'total_tokens': 49, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'compute_units': None, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_c721606bd7', 'id': 'chatcmpl-EJaDW31oRA8cB4QZLXtHIoQb6ugC0', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a06124-35be-7a81-a542-e91d563b4114-0', tool_calls=[], invalid_tool_c